<a href="https://colab.research.google.com/github/ahmedela2005-lgtm/project-/blob/main/MapReduce_Bonus_Colab_Explained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import re
import zlib

input_dir = Path('input')
output_dir = Path('output')
input_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

sample_files = {
    'split1.txt': 'big data big python map reduce',
    'split2.txt': 'big data science map python',
    'split3.txt': 'data science big reduce data'
}
for filename, content in sample_files.items():
    (input_dir / filename).write_text(content, encoding='utf-8')

files = sorted(input_dir.glob('*.txt'))
for path in files:
    print(path.name, '=>', path.read_text(encoding='utf-8'))
print('Number of splits:', len(files))

split1.txt => big data big python map reduce
split2.txt => big data science map python
split3.txt => data science big reduce data
Number of splits: 3


In [6]:
def mapper(path):
    text = path.read_text(encoding='utf-8').lower()
    words = re.findall(r'[a-z]+', text)  # simple English word tokenizer
    return path.name, [(word, 1) for word in words]

print('Example of one mapper:')
print(mapper(files[0]))

Example of one mapper:
('split1.txt', [('big', 1), ('data', 1), ('big', 1), ('python', 1), ('map', 1), ('reduce', 1)])


In [7]:
with ThreadPoolExecutor(max_workers=len(files)) as pool:
    mapper_results = list(pool.map(mapper, files))

for filename, pairs in mapper_results:
    print(f'{filename}: {pairs}')

split1.txt: [('big', 1), ('data', 1), ('big', 1), ('python', 1), ('map', 1), ('reduce', 1)]
split2.txt: [('big', 1), ('data', 1), ('science', 1), ('map', 1), ('python', 1)]
split3.txt: [('data', 1), ('science', 1), ('big', 1), ('reduce', 1), ('data', 1)]


In [8]:
def combiner(pairs):
    return Counter(word for word, one in pairs)

def partitioner(word):
    return zlib.crc32(word.encode('utf-8')) % 2

partitions = [[], []]  # 2 reducers
for filename, pairs in mapper_results:
    local_counts = combiner(pairs)
    print('COMBINE', filename, '=>', dict(sorted(local_counts.items())))
    for word, count in local_counts.items():
        reducer_id = partitioner(word)
        partitions[reducer_id].append((word, count))

for i, items in enumerate(partitions):
    print(f'PARTITION for reducer {i}:', items)

COMBINE split1.txt => {'big': 2, 'data': 1, 'map': 1, 'python': 1, 'reduce': 1}
COMBINE split2.txt => {'big': 1, 'data': 1, 'map': 1, 'python': 1, 'science': 1}
COMBINE split3.txt => {'big': 1, 'data': 2, 'reduce': 1, 'science': 1}
PARTITION for reducer 0: [('python', 1), ('reduce', 1), ('science', 1), ('python', 1), ('science', 1), ('reduce', 1)]
PARTITION for reducer 1: [('big', 2), ('data', 1), ('map', 1), ('big', 1), ('data', 1), ('map', 1), ('data', 2), ('big', 1)]


In [9]:
def shuffle_and_sort(items):
    grouped = defaultdict(list)
    for word, count in items:
        grouped[word].append(count)
    return dict(sorted(grouped.items()))

grouped_partitions = [shuffle_and_sort(p) for p in partitions]
for i, grouped in enumerate(grouped_partitions):
    print(f'SHUFFLE & SORT reducer {i}:', grouped)

SHUFFLE & SORT reducer 0: {'python': [1, 1], 'reduce': [1, 1], 'science': [1, 1]}
SHUFFLE & SORT reducer 1: {'big': [2, 1, 1], 'data': [1, 1, 2], 'map': [1, 1]}


In [10]:
def reducer(task):
    reducer_id, grouped = task
    final_counts = {word: sum(values) for word, values in grouped.items()}
    output_file = output_dir / f'part-{reducer_id:05d}.txt'
    output_file.write_text(
        ''.join(f'{word}\t{count}\n' for word, count in final_counts.items()),
        encoding='utf-8'
    )
    return reducer_id, final_counts, output_file

with ThreadPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(reducer, enumerate(grouped_partitions)))

merged = {}
for reducer_id, counts, path in results:
    merged.update(counts)
    print(f'REDUCER {reducer_id} -> {path}')
    print(path.read_text(encoding='utf-8'))

print('FINAL WORD COUNTS:', dict(sorted(merged.items())))

REDUCER 0 -> output/part-00000.txt
python	2
reduce	2
science	2

REDUCER 1 -> output/part-00001.txt
big	4
data	4
map	2

FINAL WORD COUNTS: {'big': 4, 'data': 4, 'map': 2, 'python': 2, 'reduce': 2, 'science': 2}


In [11]:
expected = Counter()
for path in files:
    expected.update(re.findall(r'[a-z]+', path.read_text(encoding='utf-8').lower()))
assert dict(expected) == merged, 'Error: output differs from original input!'
assert len(list(output_dir.glob('part-*.txt'))) == 2
print('PASS: MapReduce counts are correct.')
print('Total input words:', sum(expected.values()))
print('Total unique words:', len(expected))
print('Output files:', [p.name for p in sorted(output_dir.glob('part-*.txt'))])

PASS: MapReduce counts are correct.
Total input words: 16
Total unique words: 6
Output files: ['part-00000.txt', 'part-00001.txt']


In [12]:
import shutil
archive_path = shutil.make_archive('mapreduce_results', 'zip', '.', 'output')
print('Created:', archive_path)
try:
    from google.colab import files as colab_files
    colab_files.download(archive_path)
except ImportError:
    print('Not on Colab: find the archive in your current working folder.')

Created: /content/mapreduce_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>